# Validator Agent Notebook

This notebook demonstrates the Validator Agent which validates quantum circuits and uses LLM to fix broken code.

## Features
- **Remote Validation**: Uses QCanvas backend API for code execution
- **LLM-Powered Fixing**: When code fails, the LLM analyzes errors and suggests fixes
- **Local Fallback**: Can also run in local mode using Braket directly

## Prerequisites
1. QCanvas backend running at `http://localhost:8000`
2. Ollama with `braket-validator-agent` model created

---

# Part 1: Setup & Backend Connection Test

Before starting validation, we need to verify the backend is accessible.

In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.braket_rag_code_assistant.config import get_config, setup_logging
from src.agents.validator import ValidatorAgent
from src.tools.qcanvas_client import QCanvasClient

# Setup logging
setup_logging()
print("✅ Imports completed successfully.")

2025-12-07 15:45:11 | INFO     | src.braket_rag_code_assistant.config.logging:setup_all:138 | Logging configuration completed


✅ Imports completed successfully.


## 1.1 Check Backend Health

Verify the QCanvas backend is running and accessible.

In [2]:
# Initialize QCanvas client
client = QCanvasClient()
print(f"Backend URL: {client.base_url}")
print(f"Timeout: {client.timeout}s")
print()

# Check health
backend_healthy = client.check_health()

if backend_healthy:
    print("✅ QCanvas backend is HEALTHY!")
    frameworks = client.get_supported_frameworks()
    print(f"   Supported frameworks: {frameworks}")
else:
    print("❌ QCanvas backend is NOT AVAILABLE!")
    print("")
    print("Please start the backend:")
    print("  cd ../backend")
    print("  python start.py")

2025-12-07 15:45:11 | INFO     | config.config_loader:load:93 | ✅ Loaded configuration from D:\University\Uni\Semester 7\Generative AI\Project\Braket-RAG-Code-Assistant\config\config.json
2025-12-07 15:45:11 | INFO     | src.tools.qcanvas_client:__init__:33 | QCanvasClient initialized with URL: http://localhost:8000


Backend URL: http://localhost:8000
Timeout: 30s

✅ QCanvas backend is HEALTHY!
   Supported frameworks: ['qiskit', 'braket', 'pennylane']


## 1.2 Test Backend API: Convert Code to QASM

Test the `/api/converter/convert` endpoint with sample Braket code.

In [3]:
# Test code for conversion
test_code = """
from braket.circuits import Circuit

# Simple Bell state circuit
q0, q1 = braket.LineQubit.range(2)
circuit = braket.Circuit(
    braket.H(q0),
    braket.CNOT(q0, q1),
    braket.measure(q0, q1, key='m')
)
"""

print("📤 Sending code to /api/converter/convert...")
print()

conv_result = client.convert_to_qasm(test_code, framework="braket")

print(f"Success: {conv_result.get('success')}")
print()

if conv_result.get('success'):
    print("✅ Conversion successful!")
    print()
    print("📄 Generated OpenQASM 3.0:")
    print("-" * 40)
    qasm_code = conv_result.get('qasm_code', '')
    print(qasm_code)
    print("-" * 40)
    
    stats = conv_result.get('conversion_stats', {})
    if stats:
        print(f"\nConversion Stats: {stats}")
else:
    print("❌ Conversion failed!")
    print(f"Error: {conv_result.get('error')}")

📤 Sending code to /api/converter/convert...



2025-12-07 15:45:17 | INFO     | src.tools.qcanvas_client:convert_to_qasm:61 | Code converted successfully to QASM


Success: True

✅ Conversion successful!

📄 Generated OpenQASM 3.0:
----------------------------------------
OPENQASM 3.0;
include "stdgates.inc";


// Quantum and classical registers
qubit[2] q;
bit[2] c;

// Circuit operations
h q[0];
cx q[0], q[1];
c[0] = measure q[0];
c[1] = measure q[1];
----------------------------------------

Conversion Stats: {'qubits': 2, 'gates': {'h': 1, 'cx': 1}, 'depth': 2, 'conversion_time': None}


## 1.3 Test Backend API: Execute QASM Simulation

Test the `/api/simulator/execute-qsim` endpoint with the generated QASM.

In [4]:
# Only run if conversion was successful
if conv_result.get('success') and conv_result.get('qasm_code'):
    qasm_code = conv_result['qasm_code']
    
    print("📤 Sending QASM to /api/simulator/execute-qsim...")
    print(f"   Backend: braket")
    print(f"   Shots: 1024")
    print()
    
    sim_result = client.execute_qasm(qasm_code, backend="braket", shots=1024)
    
    print(f"Success: {sim_result.get('success')}")
    print()
    
    if sim_result.get('success'):
        print("✅ Simulation successful!")
        print()
        
        results = sim_result.get('results', {})
        
        print("📊 SIMULATION RESULTS:")
        print("-" * 40)
        print(f"Counts: {results.get('counts', {})}")
        print(f"Probabilities: {results.get('probs', {})}")
        print()
        
        metadata = results.get('metadata', {})
        if metadata:
            print("📈 Metadata:")
            print(f"   Execution Time: {metadata.get('execution_time', 'N/A')}")
            print(f"   Simulation Time: {metadata.get('simulation_time', 'N/A')}")
            print(f"   N Qubits: {metadata.get('n_qubits', 'N/A')}")
            print(f"   Fidelity: {metadata.get('fidelity', 'N/A')}%")
            print(f"   Memory Usage: {metadata.get('memory_usage', 'N/A')}")
    else:
        print("❌ Simulation failed!")
        print(f"Error: {sim_result.get('error')}")
else:
    print("⚠️ Skipping simulation test - conversion was not successful.")

📤 Sending QASM to /api/simulator/execute-qsim...
   Backend: braket
   Shots: 1024



2025-12-07 15:45:19 | INFO     | src.tools.qcanvas_client:execute_qasm:98 | QASM executed successfully. Counts: {'00': 520, '11': 504}


Success: True

✅ Simulation successful!

📊 SIMULATION RESULTS:
----------------------------------------
Counts: {'00': 520, '11': 504}
Probabilities: None

📈 Metadata:
   Execution Time: 4.87ms
   Simulation Time: 4.87ms
   N Qubits: 2
   Fidelity: 100.0%
   Memory Usage: 0.10MB


## 1.4 Test Full Pipeline: validate_and_execute()

Test the combined convert + execute pipeline.

In [5]:
print("📤 Testing full pipeline (convert + execute)...")
print()

full_result = client.validate_and_execute(test_code, shots=1024)

print(f"Success: {full_result.get('success')}")
print(f"Stage: {full_result.get('stage')}")
print()

if full_result.get('success'):
    print("✅ Full pipeline successful!")
    results = full_result.get('results', {})
    print(f"   Counts: {results.get('counts', {})}")
else:
    print("❌ Pipeline failed at stage:", full_result.get('stage'))
    print(f"   Error: {full_result.get('error')}")

print()
print("=" * 50)
print("🎉 BACKEND TEST COMPLETE - Ready for Validation!")
print("=" * 50)

📤 Testing full pipeline (convert + execute)...



2025-12-07 15:45:21 | INFO     | src.tools.qcanvas_client:convert_to_qasm:61 | Code converted successfully to QASM
2025-12-07 15:45:23 | INFO     | src.tools.qcanvas_client:execute_qasm:98 | QASM executed successfully. Counts: {'00': 527, '11': 497}


Success: True
Stage: simulation

✅ Full pipeline successful!
   Counts: {'00': 527, '11': 497}

🎉 BACKEND TEST COMPLETE - Ready for Validation!


---

# Part 2: Validator Agent Testing

Now that the backend is verified, let's test the ValidatorAgent.

## 2.1 Initialize Validator Agent

In [6]:
# First, add imports and setup RAG components
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever

# Initialize Knowledge Base with all datasets
KNOWLEDGE_BASE_DIR = project_root / "data" / "knowledge_base"
kb = KnowledgeBase(knowledge_base_path=KNOWLEDGE_BASE_DIR)
kb.load_from_directory()
kb.index_entries()
print(f"Loaded {len(kb.entries)} KB entries for semantic validation")

# Create retriever
retriever = Retriever(knowledge_base=kb)

# Initialize in remote mode with RAG retriever
validator = ValidatorAgent(retriever=retriever)  # Mode is read from config.json

print("✅ ValidatorAgent initialized with RAG support")
...
print(f"   Mode: {validator.mode}")
print(f"   LLM Enabled: {validator.llm_enabled}")
print(f"   Ollama Model: {validator.ollama_model}")
print(f"   Default Shots: {validator.default_shots}")


2025-12-07 15:45:23 | INFO     | src.rag.embeddings:__init__:97 | Loading embedding model: BAAI/bge-base-en-v1.5
2025-12-07 15:45:23 | INFO     | src.rag.embeddings:__init__:98 | Using device: cpu


2025-12-07 15:45:23,454 - sentence_transformers.SentenceTransformer - INFO - SentenceTransformer.py:227 - Load pretrained SentenceTransformer: BAAI/bge-base-en-v1.5


2025-12-07 15:45:27 | INFO     | src.rag.embeddings:__init__:106 | ✅ Embedding model loaded successfully
2025-12-07 15:45:27 | INFO     | src.rag.embeddings:__init__:113 | Embedding dimension: 768
2025-12-07 15:45:27 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2025-12-07 15:45:27 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2025-12-07 15:45:27 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 768
2025-12-07 15:45:27 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2025-12-07 15:45:27 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 | Loading knowledge base from D:\University\Uni\Semester 7\Generative AI\Project\Braket-RAG-Code-Assistant\data\knowledge_base\curated_designer_examples.jsonl
2025-12-07 15:45:27 | INFO     | src.rag.knowledge_base:load_from_jsonl:129 | Loaded 107 entries from D:\University\Uni\Semester 7\Generative AI\Project\B

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

2025-12-07 15:45:46 | INFO     | src.rag.knowledge_base:index_entries:248 | ✅ Indexed 140 entries
2025-12-07 15:45:46 | INFO     | src.rag.retriever:__init__:170 | Initialized Retriever with top_k=5, threshold=0.7, topic_boost=0.15, hybrid_scoring=True
2025-12-07 15:45:46 | INFO     | src.agents.base_agent:__init__:78 | Initialized ValidatorAgent agent
2025-12-07 15:45:46 | INFO     | src.tools.simulator:__init__:82 | Initialized simulator simulator
2025-12-07 15:45:46 | INFO     | src.agents.validator:__init__:94 | ValidatorAgent initialized in 'local' mode (RAG: enabled)


Loaded 140 KB entries for semantic validation
✅ ValidatorAgent initialized with RAG support
   Mode: local
   LLM Enabled: True
   Ollama Model: braket-validator-agent
   Default Shots: 1024


## 2.2 Validate Valid Code (Bell State)

In [7]:
valid_code = """
from braket.circuits import Circuit

# Create a Bell state circuit
q0, q1 = braket.LineQubit.range(2)
circuit = braket.Circuit(
    braket.H(q0),
    braket.CNOT(q0, q1),
    braket.measure(q0, q1, key='result')
)
print(circuit)
"""

task = {
    "code": valid_code,
    "description": "Create a Bell state with two entangled qubits",
    "shots": 1024
}

print("🔍 Validating Bell state code...")
result = validator.execute(task)

print(f"\n{'✅' if result.get('success') else '❌'} Validation {'PASSED' if result.get('success') else 'FAILED'}")
print(f"Stage: {result.get('stage')}")

if result.get('success'):
    results = result.get('results', {})
    print(f"\n📊 Results:")
    print(f"   Counts: {results.get('counts', {})}")
    print(f"   Probabilities: {results.get('probs', {})}")
else:
    print(f"\n🔴 Error: {result.get('error')}")

2025-12-07 15:45:46 | INFO     | src.tools.simulator:simulate:147 | Simulation completed. Histogram: {'11': 513, '00': 511}


🔍 Validating Bell state code...

✅ Validation PASSED
Stage: simulation

📊 Results:
   Counts: {'11': 513, '00': 511}
   Probabilities: None


## 2.3 Validate Invalid Code (with LLM Fix)

In [8]:
# Code with missing import and missing measurement
invalid_code = """
# Missing from braket.circuits import Circuit!
q0, q1 = braket.LineQubit.range(2)
circuit = braket.Circuit(
    braket.H(q0),
    braket.CNOT(q0, q1)
)
"""

task = {
    "code": invalid_code,
    "description": "Create a simple quantum circuit with H and CNOT gates",
    "force_llm_fix": True
}

print("🔍 Validating broken code (missing import)...")
result = validator.execute(task)

print(f"\n{'✅' if result.get('success') else '❌'} Validation {'PASSED' if result.get('success') else 'FAILED'}")
print(f"Stage: {result.get('stage')}")

if result.get('error'):
    print(f"\n🔴 Error: {result.get('error')}")

# Show LLM analysis
llm_analysis = result.get('llm_analysis', {})
if llm_analysis:
    print("\n🤖 LLM Analysis:")
    if llm_analysis.get('fixed_code'):
        print("\n📝 Fixed Code:")
        print("```python")
        print(llm_analysis['fixed_code'])
        print("```")
    if llm_analysis.get('analysis'):
        print(f"\n💡 Explanation: {llm_analysis['analysis']}")

2025-12-07 15:45:47 | ERROR    | src.tools.simulator:simulate:158 | Simulation error: Circuit has no measurements to sample.
2025-12-07 15:45:47 | WARNING  | src.agents.validator:_execute_local:438 | Simulation failed: Circuit has no measurements to sample.
2025-12-07 15:45:47 | INFO     | src.agents.validator:_execute_local:496 | Running LLM analysis for code fixing...


🔍 Validating broken code (missing import)...

✅ Validation PASSED
Stage: simulation_failed

🔴 Error: Circuit has no measurements to sample.

🤖 LLM Analysis:

📝 Fixed Code:
```python
from braket.circuits import Circuit

# Define qubits
q0, q1 = braket.LineQubit.range(2)

# Create a circuit with H and CNOT gates
circuit = braket.Circuit(
    braket.H(q0),
    braket.CNOT(q0, q1)
)

# Add a measurement to the circuit
circuit.append(braket.measure(q0, q1))

# Simulate the circuit
simulator = braket.Simulator()
result = simulator.run(circuit, repetitions=1000)

# Print the results
print(result.histogram(key='q(0),q(1)'))
```

💡 Explanation: **Description:**
The original code lacked a measurement operation, which is necessary for sampling and obtaining results from a quantum circuit. I added a `braket.measure(q0, q1)` to measure both qubits at the end of the circuit. This allows the simulator to produce meaningful output when run.


## 2.4 Re-validate Fixed Code

In [9]:
fixed_code = result.get('fixed_code') or result.get('llm_analysis', {}).get('fixed_code')

if fixed_code:
    print("🔍 Re-validating LLM-fixed code...")
    
    task_fixed = {
        "code": fixed_code,
        "description": "Fixed version of the circuit",
        "shots": 1024
    }
    
    result_fixed = validator.execute(task_fixed)
    
    print(f"\n{'✅' if result_fixed.get('success') else '❌'} Fixed code {'PASSED' if result_fixed.get('success') else 'FAILED'}")
    
    if result_fixed.get('success'):
        results = result_fixed.get('results', {})
        print(f"   Counts: {results.get('counts', {})}")
    else:
        print(f"   Error: {result_fixed.get('error')}")
else:
    print("⚠️ No fixed code available from previous step.")

2025-12-07 15:46:01 | INFO     | src.tools.simulator:simulate:147 | Simulation completed. Histogram: {'11': 534, '00': 490}


🔍 Re-validating LLM-fixed code...

✅ Fixed code PASSED
   Counts: {'11': 534, '00': 490}


---
# Part 3: Comprehensive Test Suite

Run the full suite of 20+ test cases to verify the Validator's robustness.

In [10]:
from tests.test_validator_circuits import run_tests

# Run all test cases
df_results = run_tests(validator)

# Display summary
print("\n" + "="*40)
print("TEST SUITE SUMMARY")
print("="*40)
print(f"Total Tests: {len(df_results)}")
print(f"Passed: {len(df_results[df_results['Status'].str.contains('PASS')])}")
print(f"Failed: {len(df_results[df_results['Status'].str.contains('FAIL|ERROR')])}")

# Show full table
df_results

📝 Logging execution details to: D:\University\Uni\Semester 7\Generative AI\Project\Braket-RAG-Code-Assistant\tests\test_execution.log
🚀 Starting Validator Test Suite (22 cases)...

[1/22] Testing: Teleportation - Valid... 

15:46:01 | Simulation completed. Histogram: {'1': 550, '0': 474}
15:46:01 | Running LLM analysis for code fixing...


-> PASS (8.25s)
[2/22] Testing: Teleportation - Missing Import... -> PASS (Detected) (0.00s)
[3/22] Testing: Teleportation - Typo in Module... -> PASS (Detected) (0.00s)
[4/22] Testing: Teleportation - Undefined Variable... -> PASS (Detected) (0.00s)
[5/22] Testing: Teleportation - Syntax Error... -> PASS (Detected) (0.00s)
[6/22] Testing: Deutsch-Jozsa - Valid... 

15:46:09 | Simulation completed. Histogram: {'1': 1024}
15:46:09 | Running LLM analysis for code fixing...


-> PASS (8.63s)
[7/22] Testing: Deutsch-Jozsa - Indentation Error... -> PASS (Detected) (0.00s)
[8/22] Testing: Deutsch-Jozsa - Wrong Gate Usage... -> PASS (Detected) (0.00s)
[9/22] Testing: Deutsch-Jozsa - Logic/Type Error... -> PASS (Detected) (0.00s)
[10/22] Testing: Deutsch-Jozsa - Missing Measurement... 

...

15:46:30 | Simulation completed. Histogram: {'0': 1024}
15:46:30 | Running LLM analysis for code fixing...


-> PASS (6.93s)
[16/22] Testing: Grover - Valid... 

15:46:37 | Simulation completed. Histogram: {'1': 1024}
15:46:37 | Running LLM analysis for code fixing...


-> PASS (8.91s)
[17/22] Testing: Grover - Argument Mismatch... -> PASS (Detected) (0.00s)
[18/22] Testing: Grover - Variable Name Typo... -> PASS (Detected) (0.00s)
[19/22] Testing: Grover - Unsupported Gate... -> PASS (Detected) (0.00s)
[20/22] Testing: Grover - Invalid Append Type... -> PASS (Detected) (0.00s)
[21/22] Testing: Misc - Empty Code... -> PASS (Detected) (0.00s)
[22/22] Testing: Misc - Natural Language... -> PASS (Detected) (0.00s)

TEST SUITE SUMMARY
Total Tests: 22
Passed: 20
Failed: 2


,ID,Name,Description,Expected,Actual_Success,Has_Fix,Status,Error_Msg
0,1,Teleportation - Valid,Standard Teleportation circuit (Separate appen...,PASS,True,True,PASS,
1,2,Teleportation - Missing Import,Using invalid module to force import error,FAIL,False,False,PASS (Detected),No circuit found in code...
2,3,Teleportation - Typo in Module,Typo 'crique' instead of 'braket',FAIL,False,False,PASS (Detected),
3,4,Teleportation - Undefined Variable,Using q3 which is not defined,FAIL,False,False,PASS (Detected),
4,5,Teleportation - Syntax Error,Missing closing parenthesis,FAIL,False,False,PASS (Detected),
5,6,Deutsch-Jozsa - Valid,Standard DJ circuit for 2 qubits (Simplified),PASS,True,True,PASS,
6,7,Deutsch-Jozsa - Indentation Error,Python indentation error in loop,FAIL,False,False,PASS (Detected),
7,8,Deutsch-Jozsa - Wrong Gate Usage,Using parameters for non-parametric gate,FAIL,False,False,PASS (Detected),
8,9,Deutsch-Jozsa - Logic/Type Error,Trying to operate on integer instead of qubit,FAIL,False,False,PASS (Detected),
9,10,Deutsch-Jozsa - Missing Measurement,Circuit has no measurements,FAIL,True,True,FAIL (False Positive),Circuit has no measurements to sample....
